In [1]:
import os
import boto3
import pickle
import pandas as pd
import numpy as np
from tqdm import tqdm

try:
    import catboost
except ModuleNotFoundError:
    ! pip install catboost

### Functions

In [2]:
# download from s3
def download_from_s3(str_local_path, str_bucket_path, str_project):
    # download file
    boto3.client('s3').download_file(str_project, str_bucket_path, str_local_path)

In [3]:
# upload to s3
def upload_to_s3(str_local_path, str_bucket_path, str_project):
    boto3.resource('s3').Bucket(str_project).Object(str_bucket_path).upload_file(str_local_path)

In [4]:
# to replace values with nan for df
def convert_to_int(str_reason):
    try:
        str_reason = int(str_reason)
        str_reason = np.nan
    except:
        pass
    return str_reason

In [5]:
# edit model
def edit_model(str_reason, str_model):
    if str_reason == 'TBD':
        return 'TBD'
    else:
        return str_model

### Constants

In [6]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
str_dirname_output = './output'
str_variant = 'noPTImodel10'

Project: 20231010-gen-xii


### Output directory

In [7]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

### Get inference models

In [8]:
# AD
str_filename = 'final_model.pkl'
str_bucket_path = f'01_ad/02_model/{str_variant}/03_final_model/{str_filename}'
str_local_path = f'./{str_filename}'
download_from_s3(
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project=str_project,
)
cls_model_inference_ad = pickle.load(open(str_local_path, 'rb'))['model_inference']
# rm
os.remove(str_local_path)

In [9]:
# PD
str_filename = 'final_model.pkl'
str_bucket_path = f'02_pricing_pd/02_model/{str_variant}/03_final_model/{str_filename}'
str_local_path = f'./{str_filename}'
download_from_s3(
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project=str_project,
)
cls_model_inference_pd = pickle.load(open(str_local_path, 'rb'))['model_inference']
# rm
os.remove(str_local_path)

In [10]:
# LGD
str_filename = 'final_model.pkl'
str_bucket_path = f'03_pricing_lgd/02_model/{str_variant}/03_final_model/{str_filename}'
str_local_path = f'./{str_filename}'
download_from_s3(
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project=str_project,
)
cls_model_inference_lgd = pickle.load(open(str_local_path, 'rb'))['model_inference']
# rm
os.remove(str_local_path)

### Combine lists of features

In [11]:
# combine lists of features
#list_cols_all = cls_model_inference_ad.feature_names_ + cls_model_inference_pd.feature_names_ + cls_model_inference_lgd.feature_names_
list_cols_all = cls_model_inference_ad.feature_names_ + cls_model_inference_pd.feature_names_
print(f'Before removing duplicates, there were {len(list_cols_all)} features in all models')
# rm dups
list_cols_all = list(dict.fromkeys(list_cols_all))
print(f'After removing duplicates, there are {len(list_cols_all)} features in all models')

Before removing duplicates, there were 335 features in all models
After removing duplicates, there are 286 features in all models


### Load reasons

In [12]:
str_filename = 'df_aa_prod.csv'
str_uri = f's3://{str_project}/ad_hoc/make_adverse_action_dictionary/{str_variant}/{str_filename}'
df_aa = pd.read_csv(str_uri)
# make col
#df_aa['model'] = 'gen_xii'
df_aa

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:275: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


,feature,reason,model,description,Change Reason/Comment
0,inquiryshortterm12month__ln,Excessive inquiries,gen_xii,Indicates subject has one or more LexisNexis p...,NaN
1,bkc84__tu,Excessive Account Balances,gen_xii,Months since a Bankcard trade last exceeded 90...,NaN
2,linkf165__tu,Excessive number of sources for phone number o...,gen_xii,Number of distinct home phones within last 18 ...,NaN
3,rt21s__tu,"Insufficient credit file, Length of Credit",gen_xii,Months since most recent retail trade opened,NaN
4,lienjudgmenttaxcount__ln,Tax or other liens,gen_xii,"Total federal, state, and county tax liens on ...",NaN
...,...,...,...,...,...
522,per222__tu,Excessive Account Balances,genxii_v2,Average total Payment Ratio for Personal Loan ...,NaN
523,rev201__tu,Excessive Account Balances,genxii_v2,Total Payment Ratio for Revolving accounts ove...,NaN
524,linka005__tu,Derogatory Public Records,genxii_v2,Average number of checks ordered in the last 3...,NaN
525,fr02s__tu,Excessive number of accounts,genxii_v2,Number of open finance revolving trades,NaN


### Tag if in AD or PD model

In [13]:
df_aa['in_model'] = df_aa['feature'].apply(
    lambda x: 1 if x in list_cols_all else 0,
)
df_aa = df_aa[df_aa['in_model'] == 1].copy()
df_aa

,feature,reason,model,description,Change Reason/Comment,in_model
0,inquiryshortterm12month__ln,Excessive inquiries,gen_xii,Indicates subject has one or more LexisNexis p...,NaN,1
1,bkc84__tu,Excessive Account Balances,gen_xii,Months since a Bankcard trade last exceeded 90...,NaN,1
2,linkf165__tu,Excessive number of sources for phone number o...,gen_xii,Number of distinct home phones within last 18 ...,NaN,1
5,linka015__tu,Excessive Bank Account Closures,gen_xii,Number of checking account closures for cause ...,NaN,1
7,ret84__tu,Excessive Account Balances,gen_xii,Months since a Retail trade last exceeded 90% ...,NaN,1
...,...,...,...,...,...,...
499,ret205__tu,Excessive Account Balances,genxii_v2,Total Payment Ratio for Retail accounts over t...,NaN,1
500,at25s__tu,Delinquent credit obligations,genxii_v2,Number of currently open and satisfactory trad...,NaN,1
501,rev315__tu,High revolving credit balances,genxii_v2,Number of months since Max Total Open-to-Buy f...,NaN,1
502,st25s__tu,Excessive Account Balances,genxii_v2,Number of currently open and satisfactory stud...,NaN,1


### Find the features with no reason

In [14]:
list_cols_no_reason = [col for col in list_cols_all if col not in list(df_aa['feature'])]
print(f'There are {len(list_cols_no_reason)} features with no reason:')
for a, col in enumerate(list_cols_no_reason):
    print(f'{a+1} - {col}')

There are 11 features with no reason:
1 - au57s__tu
2 - co05s__tu
3 - linkf002__tu
4 - rvdex01__tu
5 - alertregulatorycondition__ln
6 - addrinputavmvalue12month__ln
7 - linkt002__tu
8 - ENG-vehicle_age
9 - addrcurrentsubjectowned__ln
10 - addrpreviouslengthofres__ln
11 - g206b__tu


### Get the features that have no reason

In [15]:
# df_aa_tmp = df_aa[df_aa['reason'] == 'TBD'].copy()
# list_cols = list(df_aa_tmp['feature'])
# print(f'There are {len(list_cols)} features with no reason')

### Load Gen XI adverse action dictionary

In [16]:
# str_filename = 'cls_parse_payload_with_aa.pkl'
# str_local_path = f'./{str_filename}'
# str_bucket_path = f'ad_hoc/make_adverse_action_dictionary/input/{str_filename}'
# download_from_s3(
#     str_local_path=str_local_path, 
#     str_bucket_path=str_bucket_path, 
#     str_project=str_project,
# )
# # import
# dict_aa_old = pickle.load(open(str_local_path, 'rb')).dict_aa_pd
# # rm
# os.remove(str_local_path)

# # replace tuaccept and tucvlink with tu
# dict_aa_old_replaced = {}
# for key, val in dict_aa_old.items():
#     # logic
#     if 'tuaccept' in key:
#         key_new = key.replace('tuaccept', 'tu')
#     elif 'tucvlink' in key:
#         key_new = key.replace('tucvlink', 'tu')
#     else:
#         key_new = key    
#     # assign
#     dict_aa_old_replaced[key_new] = val
# # show
# #dict_aa_old_replaced

### Make AA dict using Gen 11 reasons

In [17]:
# list_dict_row = []
# for col in tqdm(list_cols):
#     # get reason
#     try:
#         str_reason = dict_aa_old_replaced[col]
#     except KeyError:
#         str_reason = 'TBD'
#     # make row
#     dict_row = {
#         'feature': col,
#         'reason': str_reason,
#         'model': 'gen_xi',
#     }
#     # append
#     list_dict_row.append(dict_row)
# # make df
# df_aa_new = pd.DataFrame(list_dict_row)
# df_aa_new   

### Get the features with no reason

In [18]:
# df_aa_new_tmp = df_aa_new[df_aa_new['reason'] == 'TBD']
# list_cols = list(df_aa_new_tmp['feature'])
# print(f'There are {len(list_cols)} features with no reason')

### Concat

In [19]:
# list_df = [
#     df_aa[df_aa['reason'] != 'TBD'].copy(),
#     df_aa_new,
# ]
# df_aa_all = pd.concat(list_df)
# # show
# df_aa_all

### Replace

In [20]:
# df_aa_all['model'] = df_aa_all.apply(
#     lambda x: edit_model(
#         str_reason=x['reason'], 
#         str_model=x['model'],
#     ),
#     axis=1,
# )
# # show
# df_aa_all

### Value counts

In [21]:
# ser_freq = df_aa_all['model'].value_counts()
# df_freq = ser_freq.reset_index()
# df_freq.columns = ['feature','frequency']
# df_freq

### Get the definition from the data dictionary

In [22]:
# str_filename = 'data_dictionary.csv'
# str_local_path = f'./input/{str_filename}'
# df_tmp = pd.read_csv(str_local_path)
# dict_map = dict(zip(df_tmp['feature_name'], df_tmp['Description']))
# # map
# df_aa_all['description'] = df_aa_all['feature'].map(dict_map)

# # sort
# df_aa_all.sort_values(by='model', ascending=False, inplace=True)
# # save
# str_filename = 'df_aa_all.csv'
# str_local_path = f'{str_dirname_output}/{str_filename}'
# df_aa_all.to_csv(str_local_path, index=False)

# # show
# df_aa_all

### Upload to s3

In [23]:
# # no reasons
# df_tmp = pd.DataFrame({'feature': list_no_reason})
# # save
# str_filename = 'df_no_reason.csv'
# str_uri = f's3://{str_project}/ad_hoc/make_adverse_action_dictionary/{str_variant}/{str_filename}'
# df_tmp.to_csv(str_uri, index=False)
# # show
# df_tmp

In [24]:
# # pickle
# str_filename = 'dict_aa.pkl'
# str_local_path = f'./{str_filename}'
# pickle.dump(dict_aa, open(str_local_path, 'wb'))

In [25]:
# # upload
# str_bucket_path = f'ad_hoc/make_adverse_action_dictionary/{str_variant}/{str_filename}'
# upload_to_s3(
#     str_local_path=str_local_path, 
#     str_bucket_path=str_bucket_path, 
#     str_project=str_project,
# )
# # rm
# os.remove(str_local_path)

### Make df

In [26]:
# # make df
# df_aa = pd.DataFrame({
#     'feature': list(dict_aa.keys()),
#     'reason': list(dict_aa.values()),
# })

# # fill int reasons with nan
# df_aa['reason'] = df_aa['reason'].apply(convert_to_int)

# # join
# df_aa = pd.merge(
#     left=df_aa,
#     right=df_tmp,
#     on='feature',
#     how='outer',
# )

# # werite to s3
# str_filename = 'df_aa.csv'
# str_uri = f's3://{str_project}/ad_hoc/make_adverse_action_dictionary/{str_variant}/{str_filename}'
# df_aa.to_csv(str_uri, index=False)

# # show
# df_aa